# Hidden Markov Models for Named Entity Recognition (NER)

In this notebook we apply an HMM to **Named Entity Recognition (NER)** using the CoNLL 2002 corpus in Spanish.

Since the states (NER tags) are **known during training**, we estimate the model parameters directly by **counting** (supervised maximum likelihood) and pass them to `hmmlearn` to perform decoding with **Viterbi**.

## The model

An HMM for NER has:
- **Hidden states**: NER tags (`O`, `B-PER`, `I-PER`, `B-LOC`, ...)
- **Observations**: words in the text
- **Parameters**:
  - $\pi$: initial state probabilities
  - $A$: transition matrix between states
  - $B$: emission matrix (probability of each word given each state)

Since we know the tags, estimation is simply count and normalize.

## Setup

In [ ]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install seaborn
!pip install nltk
!pip install hmmlearn

## Loading the corpus

In [ ]:
import nltk
nltk.download('conll2002', quiet=True)

from nltk.corpus import conll2002

train_sents = list(conll2002.iob_sents('esp.train'))
test_sents  = list(conll2002.iob_sents('esp.testa'))

print(f'Training sentences: {len(train_sents)}')
print(f'Test sentences:     {len(test_sents)}')
print()
print('Example sentence (token, POS, NER tag):')
print(train_sents[0])

Each sentence is a list of tuples `(token, POS, NER_tag)`. We keep only tokens and NER tags.

In [ ]:
def extract_tokens_labels(sents):
    tokens_per_sent = [[tok for tok, pos, ner in s] for s in sents]
    labels_per_sent = [[ner for tok, pos, ner in s] for s in sents]
    return tokens_per_sent, labels_per_sent

train_tokens, train_labels = extract_tokens_labels(train_sents)
test_tokens,  test_labels  = extract_tokens_labels(test_sents)

print("Training tokens", sum(len(sublist) for sublist in train_tokens))
print("Test tokens", sum(len(sublist) for sublist in test_tokens))

In [ ]:
from collections import Counter

all_labels   = [lab for seq in train_labels for lab in seq]
label_counts = Counter(all_labels)

print('Tags and frequencies:')
for lab, cnt in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f'  {lab:12s}  {cnt:7d}')

## Building vocabularies

To keep the model manageable, we replace infrequent words with a special `<UNK>` token.

In [ ]:
import numpy as np

MIN_FREQ = 2  # words with frequency < MIN_FREQ are replaced by the unknown word token <UNK>

word_counts = Counter(tok for seq in train_tokens for tok in seq)

vocab    = ['<UNK>'] + [w for w, c in word_counts.items() if c >= MIN_FREQ]
word2idx = {w: i for i, w in enumerate(vocab)}

labels    = sorted(set(all_labels))
label2idx = {l: i for i, l in enumerate(labels)}
idx2label = {i: l for l, i in label2idx.items()}

print(f'Vocabulary size: {len(vocab)} words')
print(f'Number of tags:  {len(labels)}')
print(f'Tags: {labels}')

## Supervised parameter estimation

With known states, the three HMM parameters are estimated by counting:
$$\pi_i = \frac{\text{\# sentences starting with state } i}{\text{\# sentences}}$$

$$A_{ij} = \frac{\text{\# transitions } i \to j}{\text{\# times state } i \text{ appears}}$$

$$B_{ik} = \frac{\text{\# times state } i \text{ emits word } k}{\text{\# times state } i \text{ appears}}$$

In [ ]:
n_states = len(labels)
n_obs    = len(vocab)

pi_counts = np.zeros(n_states)
A_counts  = np.zeros((n_states, n_states))
B_counts  = np.zeros((n_states, n_obs))

for tokens_seq, labels_seq in zip(train_tokens, train_labels):
    # Initial probabilities
    pi_counts[label2idx[labels_seq[0]]] += 1

    for t, (tok, lab) in enumerate(zip(tokens_seq, labels_seq)):
        s = label2idx[lab]
        w = word2idx.get(tok, 0)  # 0 = <UNK>

        # Emission
        B_counts[s, w] += 1

        # Transition
        if t + 1 < len(labels_seq):
            s_next = label2idx[labels_seq[t + 1]]
            A_counts[s, s_next] += 1

print('Counts computed.')
print(f'  pi_counts shape: {pi_counts.shape}')
print(f'  A_counts  shape: {A_counts.shape}')
print(f'  B_counts  shape: {B_counts.shape}')

In [ ]:
def normalize_rows(counts):
    """Normalize counts to probabilities row by row.
    Rows that sum to zero (unseen states) receive a uniform distribution."""
    totals     = counts.sum(axis=-1, keepdims=True)
    safe       = np.where(totals == 0, 1, totals)
    probs      = counts / safe
    zero_rows  = (totals.squeeze() == 0)
    probs[zero_rows] = 1.0 / counts.shape[-1]
    return probs

pi = normalize_rows(pi_counts)
A  = normalize_rows(A_counts)
B  = normalize_rows(B_counts)

print('Parameters normalized.')

### Inspecting the parameters

Before building the model, let's verify that the parameters make linguistic sense.

In [ ]:
import pandas as pd

pi_df = pd.Series(pi, index=labels).sort_values(ascending=False)
print('Initial state probabilities (pi):')
print(pi_df.to_string())

In [ ]:
A_df = pd.DataFrame(A, index=labels, columns=labels)
print('Transition matrix A (rows = origin state, columns = destination state):')
print(A_df.round(3).to_string())

In [ ]:
TOP_N = 8
print(f'Top {TOP_N} most probable words per tag:')
for lab in labels:
    s         = label2idx[lab]
    top_idx   = np.argsort(B[s])[::-1][:TOP_N]
    top_words = [vocab[i] for i in top_idx]
    print(f'  {lab:12s}: {top_words}')

## Building the model with hmmlearn

We use `CategoricalHMM`, appropriate when observations are discrete categories (words).

The key trick: `init_params=''` and `params=''` tell hmmlearn not to modify the parameters we provide: no Baum-Welch, no random initialization.

In [ ]:
from hmmlearn import hmm

model = hmm.CategoricalHMM(n_components=n_states, init_params='', params='')

# Initialize model probabilities
model.startprob_    = pi
model.transmat_     = A
model.emissionprob_ = B

print('Model built with count-estimated parameters.')
print(f'  Number of states:  {model.n_components}')
print(f'  Vocabulary size:   {model.emissionprob_.shape[1]}')

## Decoding with Viterbi

Given the trained model, for each test sentence we find the most probable tag sequence:

$$\hat{y} = \arg\max_y P(y \mid x)$$

`model.predict()` implements the **Viterbi algorithm** internally.

In [ ]:
def predict_sentence(tokens):
    """Return predicted tags for a sequence of tokens."""
    obs       = np.array([[word2idx.get(tok, 0)] for tok in tokens])
    state_ids = model.predict(obs)
    return [idx2label[s] for s in state_ids]

# Example on the first test sentence
i=0
ex_tokens = test_tokens[i]
ex_gold   = test_labels[i]
ex_pred   = predict_sentence(ex_tokens)

print(f'{"Token":20s} {"Gold":12s} {"Pred":12s}')
print('-' * 46)
for tok, gold, pred in zip(ex_tokens, ex_gold, ex_pred):
    mark = 'OK' if gold == pred else 'WRONG'
    print(f'{tok:20s} {gold:12s} {pred:12s} {mark}')

## Evaluation

In [ ]:
all_gold = []
all_pred = []

for tokens, labels_seq in zip(test_tokens, test_labels):
    all_gold.extend(labels_seq)
    all_pred.extend(predict_sentence(tokens))

In [ ]:
from sklearn.metrics import classification_report

print('Full report (including O):')
print(classification_report(all_gold, all_pred, labels=labels, zero_division=0))

In [ ]:
entity_labels = [l for l in labels if l != 'O']

print('Results on entity tags (excluding O):')
print(classification_report(all_gold, all_pred, labels=entity_labels, zero_division=0))

**NOTE**: `classification_report` shows the standard retrieval metrics for each class:

- **Precision**: of all tokens predicted as class $c$, what fraction truly belong to $c$.
- **Recall**: of all tokens that truly belong to $c$, what fraction were predicted as $c$.
- **F1**: harmonic mean of precision and recall. Balances both metrics in a single number.
- **Support**: number of tokens of class $c$ in the test set.

The bottom rows summarize across classes:

- **Accuracy**: fraction of tokens correctly classified overall.
- **Micro avg**: The average of precision, recall and F1 computed globally by summing TP, FP and FN across all classes before dividing.
- **Macro avg**: unweighted average across classes: treats all classes equally regardless of frequency.
- **Weighted avg**: average weighted by support: more influenced by frequent classes.

When 'O' is excluded from labels, sklearn detects that not all classes present in the data are covered, so it skips the accuracy row (which would be meaningless for a subset of classes) and shows micro avg instead. Including 'O' makes sklearn treat it as a standard multiclass problem, so it reports accuracy normally.

In NER, the 'O' tag dominates the support, so **micro avg on entity tags only**
(excluding 'O') is usually reported to compare systems.

## Error analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_gold, all_pred, labels=labels)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels,
            cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Gold')
ax.set_title('Confusion matrix (NER with HMM)')
plt.tight_layout()
plt.show()

## Reflection

**What are the limitations of this model?**

- **Unknown words** (`<UNK>`) receive the same emission distribution regardless of context. A more sophisticated model could use morphological features (capitalization, suffixes, etc.).

- The HMM assumes that the probability of a word depends only on the current tag, not on neighboring words. More sophisticated models such as conditional random fields (CRFs) relax this assumption by allowing arbitrary features of the input.

- The model is generative: it models $P(x, y) = P(y) \cdot P(x \mid y)$. CRFs are discriminative and model $P(y \mid x)$ directly, which typically yields better results in classification tasks.